# Hard-wall correlators: $\alpha_1=3$, $20\times60$, S100

Matched comparison to the completed alpha-1 ensemble: hard/support-truncated walls at
$x=5,15$, $n_{\rm shell}=1$, $\alpha_2=30$, pure initialization,
raster-$y$, perfect correction, complex128, and **120 cycles ($2N_y$)**.
There is no soft-wall mode. This is a separate ensemble with a new seed and output folder.

At the endpoint, save each sample's squared correlator
$C_G(x,r)=\sum_{y,\mu,\nu}|C_{x,y,\mu;x,y+r,\nu}|^2/(2N_y)$,
its $x$ average, and total charge. No endpoint states, covariance matrices or spectra.
Frames/RNG are retained only in the rolling checkpoint until results finish.

Select an **A100 40-GB-class runtime**. The resident batches are **40+40+20**, limited to
30 GiB in the PyTorch allocator. This is not a measured 30-GiB occupancy target or a cap
on allocations outside PyTorch. Results are 20 immutable five-sample NPZ/JSON pairs.
Restarting skips valid pairs and resumes the active batch from a valid five-cycle checkpoint.


## 1. Mount Drive and configure

Edit paths and session controls here. The displayed scientific contract is locked for this ensemble.


In [ ]:
from pathlib import Path
import json
import torch
from google.colab import drive

drive.mount('/content/drive')
BUNDLE = '15_hard_wall_alpha3_xresolved_correlator'
REVISION = 'hard_wall_xresolved_nx20_ny60_a1-3_nsh1_s100_2ny_raster_endpoint_v1'
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs') / BUNDLE
LOCAL_BUNDLE_DIR = Path('/content') / BUNDLE
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / REVISION
SCRATCH_ROOT = Path('/content/alpha3_hard_wall_correlator_scratch')
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # set 1 to stop after one resident batch

CONFIG = {'sampling_revision': 'hard_wall_xresolved_nx20_ny60_a1-3_nsh1_s100_2ny_raster_endpoint_v1',
 'root_seed': 2026091401,
 'Nx': 20,
 'Ny_values': [60],
 'samples_per_Ny': 100,
 'execution_batch_size_by_Ny': {'60': 40},
 'result_shard_size': 5,
 'cycles_rule': '2*Ny',
 'segment_cycles': 5,
 'device': 'cuda:0',
 'dtype': 'complex128',
 'gpu_memory_hard_limit_gib': 30.0,
 'protocol': {'DW': True,
              'domain_wall_interval': [5, 15],
              'dw_truncation': True,
              'meas_slab_only': True,
              'nshell': 1,
              'alpha_1': 3.0,
              'alpha_2': 30.0,
              'filling_frac': 0.5,
              'trial_orbitals': 'X',
              'init_mode': 'default',
              'sequence': 'raster_y',
              'perfect_correction': True,
              'postselect': False,
              'postselect_probability': 0.0,
              'n_a': 0.5,
              'state_representation': 'physical_frame',
              'triv_region_local_mode': False,
              'frame_reorthonormalize_interval': 1},
 'observables': {'endpoint_only': True,
                 'cycles': '[2*Ny]',
                 'x_resolved_square_correlator': True,
                 'legacy_xavg_square_correlator': True,
                 'global_charge': True,
                 'half_filling_offset': True,
                 'full_system_covariance_materialization': False}}

if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 GPU runtime')
gpu = torch.cuda.get_device_properties(0)
if 'A100' not in gpu.name.upper() or gpu.total_memory < 38 * 1024**3:
    raise RuntimeError(f'Expected A100 40-GB-class memory, found {gpu.name}')
print(json.dumps({'contract': CONFIG, 'gpu': gpu.name, 'dtype': CONFIG['dtype'],
                  'bundle_source': str(BUNDLE_DRIVE_DIR), 'local_stage': str(LOCAL_BUNDLE_DIR),
                  'output_root': str(OUTPUT_ROOT), 'scratch_root': str(SCRATCH_ROOT),
                  'resident_batches': [40,40,20], 'durable_shards': 20, 'trajectories': 100,
                  'cycles': 120, 'sample_cycles': 12000}, indent=2), flush=True)


## 2. Stage locally and run/resume

Only the four executable files are copied from Drive, never previous data.
Run inside the notebook kernel so `tqdm.auto` displays Colab-native progress bars:
20 result shards outside, 120 physical cycles inside. Do not run a second copy against
the same output folder. Space checks inspect the mounted filesystem, not cloud quota;
keep at least 5 GiB available on Drive for checkpoint replacement.


In [ ]:
import importlib.util
import shutil
import sys

FILES = ['run_campaign.py', 'alpha3_correlator.py',
         'src/classA_U1FGTN_gpu.py', 'src/occupied_frame_gpu.py']
for relative in FILES:
    source = BUNDLE_DRIVE_DIR / relative
    if not source.is_file():
        raise FileNotFoundError(f'Missing bundle file: {source}')
    target = LOCAL_BUNDLE_DIR / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
config_path = Path('/content/alpha3_hard_wall_correlator_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2) + '\n')
runner_args = ['--config', str(config_path), '--output-root', str(OUTPUT_ROOT),
               '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY: runner_args.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    runner_args += ['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))]
module_name = '_alpha3_hard_wall_correlator_runner'
for name in (module_name, 'alpha3_correlator', 'classA_U1FGTN_gpu', 'occupied_frame_gpu'):
    sys.modules.pop(name, None)
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
print('[launch in notebook kernel]', runner_path, flush=True)
spec = importlib.util.spec_from_file_location(module_name, runner_path)
runner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = runner
spec.loader.exec_module(runner)
returncode = runner.main(runner_args)
if returncode: raise RuntimeError(f'Runner exited: {returncode}')
print('[notebook] requested work finished; inspect the completion summary above', flush=True)


## 3. Release the runtime

After the requested work finishes, release the GPU.


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
